##Objetivo
Comprender cómo las palabras funcionales pueden dominar un conteo y ocultar los temas centrales de un discurso.

#Agenda
1. Preparación
2. Medidas estadísticas
3. Identificación de palabras con mayor frecuencia de aparición
4. Eliminación de las palabras (stopwords)
5. Comparación de resultados

###1. Preparación


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
plt.style.use("seaborn-v0_8-whitegrid") # Paleta de colores pasteles
ARCHIVO = "/content/drive/MyDrive/Colab Notebooks/Fundamentos de Ciencia de Datos/Clase-7-sept/conteo_palabras_video_youtube.xlsx"
completo = pd.read_excel(ARCHIVO, sheet_name = "Conteo completo", header = 1)
depurado = pd.read_excel(ARCHIVO, sheet_name = "Conteo depurado", header = 1)
excluidas = pd.read_excel(ARCHIVO, sheet_name = "Palabras excluidas", header = 1)

for df in (completo,depurado,excluidas):
  df.columns = [str(c).strip() for c in df.columns]

completo = completo.dropna(subset = ["Palabra", "Frecuencia"]).copy()
depurado = depurado.dropna(subset = ["Palabra", "Frecuencia"]).copy()
completo["Frecuencia"] = completo["Frecuencia"].astype(int)
depurado["Frecuencia"] = depurado["Frecuencia"].astype(int)

print(f"Palabras totales: {completo['Frecuencia'].sum():,}")
print(f"Palabras únicas: {len(completo):,}")
print(f"Palabras totales depuradas: {depurado['Frecuencia'].sum():,}")
print(f"Palabras únicas depuradas: {len(depurado):,}")

display(completo.head(10))



/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Palabras totales: 4,437
Palabras únicas: 1,154
Palabras totales depuradas: 2,344
Palabras únicas depuradas: 1,063


,Palabra,Frecuencia
0,que,309
1,de,215
2,a,108
3,la,106
4,no,97
5,y,97
6,el,86
7,lo,84
8,es,71
9,en,64


### 2. Medidas estadísticas del corpus

| Medida | Definición | Utilidad |
| --- | --- | --- |
| Palabras totales (tokens) | Suma de todas las frecuencias del corpus | Tamaño total del texto en volumen de palabras |
| Palabras únicas (types) | Cantidad total de palabras distintas en el vocabulario | Tamaño del vocabulario del corpus |
| Media | Suma de frecuencias dividida entre palabras únicas | Frecuencia promedio, sensible a palabras extremadamente repetidas |
| Mediana (Q2 - 50%) | Valor central de las frecuencias ordenadas | Describe mejor una distribución no simétrica |
| Moda | Frecuencia que ocurre en más palabras | Mostrar la frecuencia más habitual (ej. muchas palabras que aparecen una vez) |
| Cuartiles (Q1, Q2, Q3) | Dividen los datos ordenados en 4 partes iguales | Localizar el 25%, 50% y 75% de las frecuencias |
| Percentiles altos (Q90, Q95) | Valores que dejan por debajo el 90% y 95% de los datos | Identificar el umbral donde comienzan las palabras muy frecuentes |
| Máximo | La mayor frecuencia registrada en el corpus | Conocer la frecuencia de la palabra más repetida |
| Rango | Diferencia entre la frecuencia máxima y la mínima | Mide la amplitud total de la distribución de frecuencias |
| Varianza muestral | Promedio de desviaciones cuadráticas con respecto a la media | Cuantifica la dispersión general de las frecuencias |
| Desviación estándar muestral | Raíz de la varianza muestral | Dispersión en las mismas unidades de frecuencia |
| IQR - Rango Intercuartílico | Diferencia entre Q3 y Q1 | Mide una dispersión más robusta del 50% central |
| Asimetría | Mide la falta de simetría en la distribución | Un valor alto revela una cola de palabras muy frecuentes |
| Curtosis exceso | Mide la concentración en las colas y el centro | Evalúa la presencia de frecuencias extremadamente altas u homogéneas |
| Diversidad léxica | Razón entre palabras únicas y palabras totales | Mide la riqueza o variedad del vocabulario usado |
| Entropía normalizada | Incertidumbre de la distribución escalada entre 0 y 1 | Evalúa qué tan uniforme o concentrado está el uso del vocabulario |
| Gini | Coeficiente de desigualdad de la distribución (0 a 1) | Un valor cercano a 1 indica alta desigualdad (pocas palabras dominan el texto) |


In [3]:
def gini(valores):
  x = np.sort(np.asrray(valores,dtype=float))
  if len(x) == 0 or x.sum()==0:
    return np.nan
  n=len(x)
  return (2*np.sum(np.arrage(1,n+1)*x)/(n*x.sum()))-(n+1)/n

def entropia_normalizada(valores):
  p = np.asarray(valores,dtype=float)
  p=p/p.sum()
  h=-(p*np.log(p)).sum()
  return h/np.log2(len(p)) if len(p)>1 else 0.0

In [4]:
def resumen_estadistico(df, nombre):
    f = df["Frecuencia"]
    modos = f.mode().tolist()
    return pd.Series({
        "Corpus": nombre,
        "Palabras totales (tokens)": int(f.sum()),
        "Palabras únicas": int(f.size),
        "Media": f.mean(),
        "Mediana": f.median(),
        "Moda": ", ".join(map(str, modos)),
        "Q1(25%)": f.quantile(.25),
        "Q2(50%)": f.quantile(.50),
        "Q3(75%)": f.quantile(.75),
        "Q90": f.quantile(.90),
        "Q95": f.quantile(.95),
        "Máximo": f.max(),
        "Rango": f.max()-f.min(),
        "Varianza muestral": f.var(ddof=1),
        "Desviación estándar muestral": f.std(ddof=1),
        "Asimetria": f.skew(),
        "Curtosis exceso": f.kurt(),
        "Diversidad léxica": f.size/f.sum(),
        "Entropía normalizada": entropia_normalizada(f),
        "Gini": gini(f),#

    })

    resumen = pd.DataFrame([
        resumen_estadistico(completo, "Completo"),
        resumen_estadistico(depurado, "Depurado"),
        resumen_estadistico(excluidas, "Excluidas")
    ]).set_index("Corpus").T
    display(resumen)